# Model Accuracy Evaluation on New Dataset
Evaluates the trained ResNet50 model on image URLs from `.txt` files (lamp_broken, scratch, dent).

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from io import BytesIO
import requests
from tqdm import tqdm
import os
from dotenv import load_dotenv

load_dotenv()

True

## Config — set paths here

In [2]:
MODEL_PATH = "/home/subhan/All/Car_Project/ImagePreprocessing/model_training_kaggle_data/notebooks/saved_training_results.pth"

# Paths to your .txt files containing image URLs
TXT_FILES = {
    "lamp_broken": "/home/subhan/All/Car_Project/FindACar/scraping/Webscraping/lamp_broken.txt",
    "scratch":     "/home/subhan/All/Car_Project/FindACar/scraping/Webscraping/scratch.txt",
    "dent":        "/home/subhan/All/Car_Project/FindACar/scraping/Webscraping/dent.txt",
}

## Load Model

In [3]:
def load_model(model_path):
    total_classes = 7
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, total_classes)
    )

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Model loaded successfully on {device}")
    return model, device


classes = {
    0: "Normal",
    1: "crack",
    2: "dent",
    3: "glass_shatter",
    4: "lamp_broken",
    5: "scratch",
    6: "tire_flat"
}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

model, device = load_model(MODEL_PATH)

Model loaded successfully on cpu


## Helper — predict a single image URL

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.pakwheels.com/"
}
session = requests.Session()


def predict_url(url):
    """Return (predicted_class_name, confidence) or (None, 0.0) on error."""
    try:
        response = session.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        input_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(input_tensor)
            probs = torch.softmax(outputs, dim=1)
            pred_index = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred_index].item()

        return classes[pred_index], confidence
    except Exception as e:
        print(f"  [ERROR] {url}: {e}")
        return None, 0.0

## Evaluate accuracy per class

In [5]:
def evaluate_txt_file(txt_path, true_label):
    """
    Read URLs from a .txt file (one per line), run prediction on each,
    and return per-image results + summary stats.
    """
    with open(txt_path, "r") as f:
        urls = [line.strip() for line in f if line.strip()]

    print(f"\n{'='*55}")
    print(f"True label : {true_label}")
    print(f"Total URLs : {len(urls)}")
    print(f"{'='*55}")

    results = []
    correct = 0
    skipped = 0

    for url in tqdm(urls, desc=f"Evaluating {true_label}"):
        predicted, confidence = predict_url(url)

        if predicted is None:
            skipped += 1
            continue

        is_correct = (predicted == true_label)
        if is_correct:
            correct += 1

        results.append({
            "url": url,
            "true_label": true_label,
            "predicted": predicted,
            "confidence": round(confidence, 4),
            "correct": is_correct
        })

    evaluated = len(urls) - skipped
    accuracy = (correct / evaluated * 100) if evaluated > 0 else 0.0

    print(f"  Evaluated : {evaluated}  |  Skipped : {skipped}")
    print(f"  Correct   : {correct}")
    print(f"  Accuracy  : {accuracy:.2f}%")

    return results, {"true_label": true_label, "total": len(urls),
                     "evaluated": evaluated, "skipped": skipped,
                     "correct": correct, "accuracy": accuracy}

In [6]:
all_results = []
summary = []

for true_label, txt_path in TXT_FILES.items():
    results, stats = evaluate_txt_file(txt_path, true_label)
    all_results.extend(results)
    summary.append(stats)


True label : lamp_broken
Total URLs : 1300


Evaluating lamp_broken: 100%|██████████| 1300/1300 [22:07<00:00,  1.02s/it]


  Evaluated : 1300  |  Skipped : 0
  Correct   : 85
  Accuracy  : 6.54%

True label : scratch
Total URLs : 1300


Evaluating scratch: 100%|██████████| 1300/1300 [08:24<00:00,  2.58it/s]


  Evaluated : 1300  |  Skipped : 0
  Correct   : 60
  Accuracy  : 4.62%

True label : dent
Total URLs : 1300


Evaluating dent: 100%|██████████| 1300/1300 [07:18<00:00,  2.97it/s]

  Evaluated : 1300  |  Skipped : 0
  Correct   : 908
  Accuracy  : 69.85%


## Overall Summary

In [7]:
import pandas as pd

summary_df = pd.DataFrame(summary)
print("\n" + "="*55)
print("OVERALL SUMMARY")
print("="*55)
print(summary_df.to_string(index=False))

total_correct  = summary_df["correct"].sum()
total_evaluated = summary_df["evaluated"].sum()
overall_acc = total_correct / total_evaluated * 100 if total_evaluated else 0

print(f"\nOverall Accuracy : {overall_acc:.2f}%  ({total_correct}/{total_evaluated})")


OVERALL SUMMARY
 true_label  total  evaluated  skipped  correct  accuracy
lamp_broken   1300       1300        0       85  6.538462
    scratch   1300       1300        0       60  4.615385
       dent   1300       1300        0      908 69.846154

Overall Accuracy : 27.00%  (1053/3900)


## Detailed results (all images)

In [8]:
results_df = pd.DataFrame(all_results)
results_df.head(20)

,url,true_label,predicted,confidence,correct
0,https://cache2.pakwheels.com/carsure_report_pi...,lamp_broken,scratch,0.3530,False
1,https://cache2.pakwheels.com/carsure_report_pi...,lamp_broken,scratch,0.3137,False
2,https://cache4.pakwheels.com/carsure_report_pi...,lamp_broken,scratch,0.4707,False
3,https://cache2.pakwheels.com/carsure_report_pi...,lamp_broken,scratch,0.7897,False
4,https://cache3.pakwheels.com/carsure_report_pi...,lamp_broken,dent,0.5657,False
5,https://cache4.pakwheels.com/carsure_report_pi...,lamp_broken,lamp_broken,0.4880,True
6,https://cache1.pakwheels.com/carsure_report_pi...,lamp_broken,dent,0.7899,False
7,https://cache2.pakwheels.com/carsure_report_pi...,lamp_broken,dent,0.6080,False
8,https://cache1.pakwheels.com/carsure_report_pi...,lamp_broken,scratch,0.8214,False
9,https://cache2.pakwheels.com/carsure_report_pi...,lamp_broken,scratch,0.8653,False


## Confusion breakdown — what did the model predict instead?

In [9]:
wrong_df = results_df[results_df["correct"] == False]
if not wrong_df.empty:
    print("Misclassification breakdown:")
    print(wrong_df.groupby(["true_label", "predicted"]).size().reset_index(name="count").to_string(index=False))
else:
    print("No misclassifications! Perfect accuracy.")

Misclassification breakdown:
 true_label     predicted  count
       dent        Normal    211
       dent         crack      8
       dent glass_shatter     60
       dent   lamp_broken     22
       dent       scratch     51
       dent     tire_flat     40
lamp_broken        Normal    169
lamp_broken         crack     72
lamp_broken          dent    479
lamp_broken glass_shatter     42
lamp_broken       scratch    339
lamp_broken     tire_flat    114
    scratch        Normal    318
    scratch         crack     12
    scratch          dent    804
    scratch glass_shatter     28
    scratch   lamp_broken     30
    scratch     tire_flat     48


## Save results to CSV (optional)

In [10]:
output_path = "/home/subhan/All/Car_Project/FindACar/evaluation_results.csv"
results_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

Results saved to: /home/subhan/All/Car_Project/FindACar/evaluation_results.csv
